# Entrenamiento — Nowcasting meteorológico
Modelo encoder (ResNet-34) + neck (ConvLSTM) + decoder (U-Net con SkipConvLSTM).  
Entrada: 4 frames de radar (t-60, t-40, t-20, t). Salida: predicción t+20 min.

**Antes de ejecutar**: asegúrate de que el entorno de ejecución tiene GPU activada:  
`Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`

## 0. Verificar GPU

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No hay GPU disponible. "
    "Ve a: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU"
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU   : {gpu_name}")
print(f"VRAM  : {vram_gb:.1f} GB")

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuración
Ajusta `DRIVE_PROJECT` a la ruta donde tienes el proyecto en tu Drive.  
Ajusta `BATCH_SIZE` según la GPU que tengas asignada:
- **T4 (16 GB)** → 64
- **A100 (40 GB)** → 128
- **V100 (16 GB)** → 64

In [ ]:
import os

# ── RUTAS ──────────────────────────────────────────────────────────────────
DRIVE_PROJECT   = "/content/drive/MyDrive/proyectoPredMet"  # <- ajusta si es necesario
SHARDS_LOCAL    = "/content/shards"
CHECKPOINT_DIR  = f"{DRIVE_PROJECT}/checkpoints"
CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/best_model.pt"

# ── HIPERPARÁMETROS ────────────────────────────────────────────────────────
H           = 128    # hidden_dim del neck ConvLSTM  (128 o 256)
DROPOUT_P   = 0.1
BATCH_SIZE  = 64     # T4 16 GB → 64  |  A100 40 GB → 128
LR          = 0.001
NUM_EPOCHES = 100
PATIENCE    = 15     # early stopping
SCHED_PAT   = 5      # épocas sin mejora antes de reducir LR a la mitad
NUM_WORKERS = 4

# ── REANUDAR desde checkpoint si ya existe ─────────────────────────────────
RESUME = True

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SHARDS_LOCAL,   exist_ok=True)
print("Configuración lista.")

## 3. Instalar dependencias

In [ ]:
%pip install -q webdataset piqa torchinfo

## 4. Copiar shards a `/content`
Leer directamente desde Drive es lento. Copiar los `.tar` a la SSD local de Colab es mucho más rápido para el DataLoader.

In [ ]:
import glob, shutil

ya_copiados = glob.glob(f"{SHARDS_LOCAL}/*.tar")
if ya_copiados:
    print(f"Shards ya presentes en /content: {len(ya_copiados)}. Omitiendo copia.")
else:
    origen = glob.glob(f"{DRIVE_PROJECT}/data/raw/shards/*.tar")
    print(f"Copiando {len(origen)} shards desde Drive...")
    for src in origen:
        shutil.copy(src, SHARDS_LOCAL)
    print("Copia completada.")

total = glob.glob(f"{SHARDS_LOCAL}/*.tar")
assert total, f"No se encontraron shards en {SHARDS_LOCAL}."
print(f"Shards listos: {len(total)}")

## 5. Imports

In [ ]:
import sys
if DRIVE_PROJECT not in sys.path:
    sys.path.insert(0, DRIVE_PROJECT)

from src.model.model import modelMet
from src.engine.training_loop import train_loop
from src.data.data_obtencion import generar_dataloaders, generar_dataset
from src.engine.loss import loss_radar

print("Imports correctos.")

## 6. Modelo, optimizador, scheduler y dataloaders

In [ ]:
device = "cuda:0"

modelo = modelMet(h=H, dropout_p=DROPOUT_P).to(device)
total_params = sum(p.numel() for p in modelo.parameters())
print(f"Parámetros totales : {total_params:,}")

optimizer = torch.optim.AdamW(
    modelo.parameters(), lr=LR, betas=(0.9, 0.999), weight_decay=0.01
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=SCHED_PAT, min_lr=1e-6
)

if RESUME and os.path.exists(CHECKPOINT_PATH):
    modelo.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    print(f"Checkpoint cargado desde: {CHECKPOINT_PATH}")
else:
    print("Entrenando desde cero.")

train_dataset, val_dataset, _ = generar_dataset(
    path_shards=f"{SHARDS_LOCAL}/*", props=[0.85, 0.1, 0.05]
)
train_loader = generar_dataloaders(
    train_dataset, split="train", batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
val_loader = generar_dataloaders(
    val_dataset, split="val", batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)

print("Todo listo.")

## 7. Entrenamiento

In [ ]:
min_loss, best_state_dict = train_loop(
    model=modelo,
    optimizer=optimizer,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    loss_module=loss_radar,
    num_epoches=NUM_EPOCHES,
    patience=PATIENCE,
    save_path=CHECKPOINT_PATH,
    scheduler=scheduler,
)

print(f"\nEntrenamiento finalizado.")
print(f"Mejor loss en validación : {min_loss:.6f}")
print(f"Modelo guardado en       : {CHECKPOINT_PATH}")